In [ ]:
import os
import csv
import ast
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib as mpl

import sys
sys.path.insert(0, str(Path("..").resolve()))
from utils import savefig

plt.rcParams['font.size'] = 14
mpl.rcParams['svg.fonttype'] = 'none'

save_fig = False

In [ ]:
df_ctx = pd.read_pickle("../data/df_ctx.pkl")
df_ctx = df_ctx[df_ctx["accuracy"] > 0.6]

In [ ]:
noise_stds = [0, 0.2, 0.4, 0.8]

In [ ]:
# bar plot of each metric for each noise std


accuracies_grouped = df_ctx.groupby("noise_std")["accuracy"]
for name, group in accuracies_grouped:
    print(name, len(group))


accuracy_mean_by_noise_std = df_ctx.groupby("noise_std")["accuracy"].mean()
accuracy_std_by_noise_std = df_ctx.groupby("noise_std")["accuracy"].std()

temporal_factor_mean_by_noise_std = df_ctx.groupby("noise_std")["temporal_factor"].mean()
temporal_factor_std_by_noise_std = df_ctx.groupby("noise_std")["temporal_factor"].std()

forward_asymmetry_mean_by_noise_std = df_ctx.groupby("noise_std")["forward_asymmetry"].mean()
forward_asymmetry_std_by_noise_std = df_ctx.groupby("noise_std")["forward_asymmetry"].std()

variance_explained_index_mean_by_noise_std = df_ctx.groupby("noise_std")["explained_variance_index"].mean()
variance_explained_index_std_by_noise_std = df_ctx.groupby("noise_std")["explained_variance_index"].std()

variance_explained_identity_mean_by_noise_std = df_ctx.groupby("noise_std")["explained_variance_identity"].mean()
variance_explained_identity_std_by_noise_std = df_ctx.groupby("noise_std")["explained_variance_identity"].std()

cross_decoding_accuracy_index_mean_by_noise_std = df_ctx.groupby("noise_std")["cross_decoding_accuracy_index"].mean()
cross_decoding_accuracy_index_std_by_noise_std = df_ctx.groupby("noise_std")["cross_decoding_accuracy_index"].std()

cross_decoding_accuracy_identity_mean_by_noise_std = df_ctx.groupby("noise_std")["cross_decoding_accuracy_identity"].mean()
cross_decoding_accuracy_identity_std_by_noise_std = df_ctx.groupby("noise_std")["cross_decoding_accuracy_identity"].std()


viridis = plt.get_cmap('viridis', lut=len(noise_stds))
colors = [viridis(i) for i in range(len(noise_stds))]

noise_std_nums = np.arange(len(noise_stds))
df_ctx["noise_std_num"] = df_ctx["noise_std"].map(lambda x: noise_std_nums[noise_stds.index(x)])

def plot_mean_scatter_bar(mean, std, df, df_label, noise_stds, y_label, save_folder=None, save_name=None):
    plt.figure(figsize=(2.7, 3.3), dpi=180)
    # plt.bar(np.arange(len(noise_stds)), mean, width=0.7, color=colors, yerr=std, capsize=5)
    plt.bar(np.arange(len(noise_stds)), mean, width=0.7, color=colors)
    plt.xticks(np.arange(len(noise_stds)), noise_stds)
    xrand = np.random.uniform(-0.2, 0.2, len(df))
    plt.scatter(df["noise_std_num"].values + xrand, df[df_label].values, color="gray", alpha=0.3, s=12)
    plt.xlabel("Context amplitude")
    plt.ylabel(y_label)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    if save_fig and save_folder is not None and save_name is not None:
        savefig(save_folder, save_name, format="svg", close=False)

    plt.show()

plot_mean_scatter_bar(accuracy_mean_by_noise_std, accuracy_std_by_noise_std, df_ctx, "accuracy", noise_stds, "Task accuracy")
plot_mean_scatter_bar(temporal_factor_mean_by_noise_std, temporal_factor_std_by_noise_std, df_ctx, "temporal_factor", noise_stds, "Temporal organization score",
                    save_folder="./figures/fig5", save_name="fig5D")
plot_mean_scatter_bar(forward_asymmetry_mean_by_noise_std, forward_asymmetry_std_by_noise_std, df_ctx, "forward_asymmetry", noise_stds, "Forward asymmetry",
                    save_folder="./figures/fig5", save_name="fig5E")
plot_mean_scatter_bar(variance_explained_index_mean_by_noise_std, variance_explained_index_std_by_noise_std, df_ctx, "explained_variance_index", noise_stds, "Variance explained by index",
                    save_folder='./figures/fig5', save_name='fig5F')
plot_mean_scatter_bar(variance_explained_identity_mean_by_noise_std, variance_explained_identity_std_by_noise_std, df_ctx, "explained_variance_identity", noise_stds, "Variance explained\nby identity")
plot_mean_scatter_bar(cross_decoding_accuracy_index_mean_by_noise_std, cross_decoding_accuracy_index_std_by_noise_std, df_ctx, "cross_decoding_accuracy_index", noise_stds, "Cross decoding accuracy\n of index",
                    save_folder='./figures/fig5', save_name='fig5G')
plot_mean_scatter_bar(cross_decoding_accuracy_identity_mean_by_noise_std, cross_decoding_accuracy_identity_std_by_noise_std, df_ctx, "cross_decoding_accuracy_identity", noise_stds, "Cross decoding accuracy\n of identity")


In [ ]:
def plot_crp_by_group(data, group_values, group_title, colors, save_folder=None, save_name=None):
    plt.figure(figsize=(4.2, 3.3), dpi=180)

    print(data.shape[1])

    mid_point = data.shape[1] // 2 + 1

    print(mid_point)
    for i in range(len(group_values)):
        plt.plot(np.arange(-mid_point+2, 0), data[i, 1:mid_point-1], label=group_values[i], color=colors[i])
        plt.plot(np.arange(1, mid_point-1), data[i, mid_point:-1], color=colors[i])
    
    plt.xlabel("Lag")
    plt.ylabel("Conditional\nrecall probability")
    plt.legend(
        title=group_title, 
        bbox_to_anchor=(0.8, 1.0), 
        loc="upper left", 
        borderaxespad=0, 
        frameon=False,
    )
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    if save_fig and save_folder is not None and save_name is not None:
        savefig(save_folder, save_name, format="svg", close=False)

    plt.show()

In [ ]:
crp_by_noise_std = df_ctx.groupby("noise_std")["crp_curve"]
# print(crp_by_noise_std.head())
mean_crp_by_noise_std = []
for name, group in crp_by_noise_std:
    crp_arrays = np.array(group.tolist())
    mean_crp = np.mean(np.stack(crp_arrays, axis=0), axis=0)
    mean_crp_by_noise_std.append(mean_crp)
mean_crp_by_noise_std = np.array(mean_crp_by_noise_std)

viridis = plt.get_cmap('viridis', lut=len(noise_stds))
colors = [viridis(i) for i in range(len(noise_stds))]

plot_crp_by_group(mean_crp_by_noise_std, noise_stds, "Context\namplitude", colors,
                save_folder="./figures/fig5", save_name="fig5B")



In [ ]:
df_samediff_raw = pd.read_pickle("../data/df_samediff.pkl")
print(len(df_samediff_raw))

In [ ]:
filter_model_nums = df_samediff_raw[(df_samediff_raw["noise_type"] == "same") & (df_samediff_raw["test_noise_type"] == "same") & (df_samediff_raw["accuracy"] > 0.5)]["model_num"].tolist()
print(filter_model_nums)
filter_model_nums2 = df_samediff_raw[(df_samediff_raw["noise_type"] == "diff") & (df_samediff_raw["test_noise_type"] == "diff") & (df_samediff_raw["accuracy"] > 0.5)]["model_num"].tolist()
print(filter_model_nums2)
df_samediff = df_samediff_raw[((df_samediff_raw["noise_type"] == "same") & (df_samediff_raw["model_num"].isin(filter_model_nums))) |
                         ((df_samediff_raw["noise_type"] == "diff") & (df_samediff_raw["model_num"].isin(filter_model_nums2))) |
                         ((df_samediff_raw["noise_type"] == "fixed") & (df_samediff_raw["accuracy"] > 0.5)) |
                         ((df_samediff_raw["noise_type"] == "none") & (df_samediff_raw["accuracy"] > 0.5))].copy()
print(len(df_samediff))

df_samediff["noise_type_int"] = df_samediff["noise_type"].map({"same": 2, "diff": 3, "fixed": 0, "none": 1})
df_samediff["test_noise_type_int"] = df_samediff["test_noise_type"].map({"same": 2, "diff": 3, "fixed": 0, "none": 1})

In [ ]:

accuracy_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["accuracy"].mean()
accuracy_std_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["accuracy"].std()
print(accuracy_mean_by_noise_type)

temporal_factor_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["temporal_factor"].mean()
temporal_factor_std_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["temporal_factor"].std()

forward_asymmetry_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["forward_asymmetry"].mean()
forward_asymmetry_std_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["forward_asymmetry"].std()

variance_explained_index_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["explained_variance_index"].mean()
variance_explained_identity_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["explained_variance_identity"].mean()

cross_decoding_accuracy_index_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["cross_decoding_accuracy_index"].mean()
cross_decoding_accuracy_identity_mean_by_noise_type = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["cross_decoding_accuracy_identity"].mean()

colors = ["#5CA4A9", "#9BC1BC", "#FF8811", "#F4D06F", "#5AA9E6", "#7FC8F8"]

def plot_mean_scatter_bar(mean, df, df_label, y_label, save_folder=None, save_name=None, figsize=(2.8, 3.3)):
    plt.figure(figsize=figsize, dpi=180)
    plt.bar(np.arange(len(mean)), mean, width=0.7, color=colors)
    plt.scatter(df["mark"], df[df_label], color="gray", alpha=0.3)
    # plt.xticks(np.arange(3), ["baseline", "same", "diff"], fontsize=12, rotation=30)
    plt.xticks([])
    plt.xlabel("Context")
    plt.ylabel(y_label)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    if save_fig and save_folder is not None and save_name is not None:
        savefig(save_folder, save_name, format="svg", close=False)

    plt.show()

# print(df_samediff["noise_type"])
plot_mean_scatter_bar(accuracy_mean_by_noise_type, df_samediff, "accuracy", "Task accuracy",
                    save_folder="./figures/exfig6", save_name="exfig6F")
plot_mean_scatter_bar(temporal_factor_mean_by_noise_type, df_samediff, "temporal_factor", "Temporal organization score",
                    save_folder="./figures/exfig6", save_name="exfig6G")
plot_mean_scatter_bar(forward_asymmetry_mean_by_noise_type, df_samediff, "forward_asymmetry", "Forward asymmetry",
                    save_folder="./figures/exfig6", save_name="exfig6H")
plot_mean_scatter_bar(variance_explained_index_mean_by_noise_type, df_samediff, "explained_variance_index", "Variance explained by index")
plot_mean_scatter_bar(variance_explained_identity_mean_by_noise_type, df_samediff, "explained_variance_identity", "Variance explained\nby identity")
plot_mean_scatter_bar(cross_decoding_accuracy_index_mean_by_noise_type, df_samediff, "cross_decoding_accuracy_index", "Cross decoding accuracy\nof index",
                    save_folder="./figures/exfig6", save_name="exfig6I", figsize=(3.0, 3.3))
plot_mean_scatter_bar(cross_decoding_accuracy_identity_mean_by_noise_type, df_samediff, "cross_decoding_accuracy_identity", "Cross decoding accuracy\nof identity")


In [ ]:
crp_by_noise_std = df_samediff.groupby(["noise_type_int", "test_noise_type_int"])["crp_curve"]
mean_crp_by_noise_std = []
for name, group in crp_by_noise_std:
    crp_arrays = np.array(group.tolist())
    mean_crp = np.mean(np.stack(crp_arrays, axis=0), axis=0)
    mean_crp_by_noise_std.append(mean_crp)
mean_crp_by_noise_std = np.array(mean_crp_by_noise_std)
colors = np.array(colors)

plot_crp_by_group(mean_crp_by_noise_std[[0,1,2,4]], ["fixed context", "no context", "matching\ncontext", "non-matching\ncontext"], "", colors[[0,1,2,4]],
                save_folder="./figures/exfig6", save_name="exfig6E")


### Semantic information

In [ ]:
df_semantic = pd.read_pickle("../data/df_semantic.pkl")
print(len(df_semantic))
semantic_amp = [0, 0.3, 0.5]

In [ ]:
crp_by_noise_std_hier = df_semantic.groupby("semantic_amp")["crp_curve"]
mean_crp_by_noise_std_hier = []
for name, group in crp_by_noise_std_hier:
    crp_arrays = np.array(group.tolist())
    mean_crp = np.mean(np.stack(crp_arrays, axis=0), axis=0)
    mean_crp_by_noise_std_hier.append(mean_crp)
mean_crp_by_noise_std_hier = np.array(mean_crp_by_noise_std_hier)

print(mean_crp_by_noise_std_hier.shape)

colors = plt.cm.viridis(np.linspace(0, 1, len(semantic_amp)))
plot_crp_by_group(mean_crp_by_noise_std_hier, semantic_amp, "Semantic\namplitude", colors,
                save_folder="./figures/exfig6", save_name="exfig6J")


In [ ]:
semantic_contiguity = df_semantic.groupby("semantic_amp")["semantic_contiguity"]
semantic_contiguity_baseline = df_semantic.groupby("semantic_amp")["semantic_contiguity_baseline"]

sem_cont_norms = []
for name, group in semantic_contiguity:
    sem_cont_norm = np.array(group.tolist())
    sem_cont_norms.append(sem_cont_norm / np.sum(sem_cont_norm, axis=1, keepdims=True))


sem_cont_norms_baseline = []
for name, group in semantic_contiguity_baseline:
    sem_cont_norm_baseline = np.array(group.tolist())
    sem_cont_norms_baseline.append(sem_cont_norm_baseline / np.sum(sem_cont_norm_baseline, axis=1, keepdims=True))


semantic_contiguity_processed = []
semantic_contiguity_processed_std = []
for i in range(len(sem_cont_norms)):
    sem_count_processed = sem_cont_norms[i] / (sem_cont_norms_baseline[i] + 1e-6)
    data_processed = []
    for j in range(len(sem_count_processed)):
        data_processed.append(sem_count_processed[j])
    data_processed = np.array(data_processed)
    print(data_processed.shape)
    semantic_contiguity_processed.append(np.mean(data_processed, axis=0))
    print(np.std(sem_cont_norms[i], axis=0), np.mean(sem_cont_norms_baseline[i], axis=0))
    semantic_contiguity_processed_std.append(np.std(sem_cont_norms[i], axis=0))

print(semantic_contiguity_processed)
print(semantic_contiguity_processed_std)

In [ ]:
plt.figure(figsize=(4, 3.3), dpi=180)
settings_name = semantic_amp
for i in range(len(semantic_contiguity_processed)):
    plt.plot(semantic_contiguity_processed[i], label=settings_name[i], color=colors[i])
    plt.errorbar(np.arange(len(semantic_contiguity_processed[i])), semantic_contiguity_processed[i], yerr=semantic_contiguity_processed_std[i], color=colors[i], capsize=2, alpha=0.5)

plt.xticks(np.arange(0, 3), [0, 1, 2])

plt.xlabel("Semantic similarity")
plt.ylabel("Relative conditional\nrecall probability")
# plt.legend(title="Semantic\namplitude", 
#         bbox_to_anchor=(1.0, 1.0), 
#         loc="upper left", 
#         borderaxespad=0, 
#         frameon=False,
#         fontsize=12)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
if save_fig:
    savefig("./figures/exfig6", "exfig6K", format="svg", close=False)

plt.show()



In [ ]:


accuracy_mean_by_noise_std = df_semantic.groupby("semantic_amp")["accuracy"].mean()
accuracy_std_by_noise_std = df_semantic.groupby("semantic_amp")["accuracy"].std()

temporal_factor_mean_by_noise_std = df_semantic.groupby("semantic_amp")["temporal_factor"].mean()
temporal_factor_std_by_noise_std = df_semantic.groupby("semantic_amp")["temporal_factor"].std()

forward_asymmetry_mean_by_noise_std = df_semantic.groupby("semantic_amp")["forward_asymmetry"].mean()
forward_asymmetry_std_by_noise_std = df_semantic.groupby("semantic_amp")["forward_asymmetry"].std()

variance_explained_index_mean_by_noise_std = df_semantic.groupby("semantic_amp")["explained_variance_index"].mean()
variance_explained_identity_mean_by_noise_std = df_semantic.groupby("semantic_amp")["explained_variance_identity"].mean()

cross_decoding_accuracy_index_mean_by_noise_std = df_semantic.groupby("semantic_amp")["cross_decoding_accuracy_index"].mean()
cross_decoding_accuracy_identity_mean_by_noise_std = df_semantic.groupby("semantic_amp")["cross_decoding_accuracy_identity"].mean()


df_semantic["mark"] = df_semantic["semantic_amp"].map(lambda x: semantic_amp.index(x)+1)

def plot_mean_scatter_bar(mean, df, df_label, y_label, x_ticks=None, figsize=(2.7, 3.3), save_folder=None, save_name=None):
    plt.figure(figsize=figsize, dpi=180)
    plt.bar(np.arange(1, len(mean)+1), mean, width=0.7, color=colors)
    plt.scatter(df["mark"], df[df_label], color="gray", alpha=0.3)
    plt.xlabel("Semantic amplitude")
    plt.xticks(np.arange(1, len(mean)+1), semantic_amp)
    plt.ylabel(y_label)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if save_fig and save_folder is not None and save_name is not None:
        savefig(save_folder, save_name, format="svg", close=False)

    plt.show()

xticks = ["Amplitude 0.2, Discount 1", "Amplitude 0.5, Discount 1", "Amplitude 0.2, Discount 0", "Amplitude 0.5, Discount 0"]
plot_mean_scatter_bar(accuracy_mean_by_noise_std, df_semantic, "accuracy", "Task accuracy", xticks)
plot_mean_scatter_bar(temporal_factor_mean_by_noise_std, df_semantic, "temporal_factor", "Temporal organization score", xticks,
                        save_folder="./figures/exfig6", save_name="exfig6L")
plot_mean_scatter_bar(forward_asymmetry_mean_by_noise_std, df_semantic, "forward_asymmetry", "Forward asymmetry", xticks,
                        save_folder="./figures/exfig6", save_name="exfig6M")
plot_mean_scatter_bar(variance_explained_index_mean_by_noise_std, df_semantic, "explained_variance_index", "Variance explained by index", xticks)
plot_mean_scatter_bar(variance_explained_identity_mean_by_noise_std, df_semantic, "explained_variance_identity", "Variance explained\nby identity", xticks)
plot_mean_scatter_bar(cross_decoding_accuracy_index_mean_by_noise_std, df_semantic, "cross_decoding_accuracy_index", "Cross decoding accuracy\n of index", xticks,
                    save_folder="./figures/exfig6", save_name="exfig6N", figsize=(2.9, 3.3))
plot_mean_scatter_bar(cross_decoding_accuracy_identity_mean_by_noise_std, df_semantic, "cross_decoding_accuracy_identity", "Cross decoding accuracy\n of identity", xticks)

